In [ ]:
# ## 1. 환경 준비
# 필요한 패키지가 설치되어 있지 않으면 아래 셀을 실행하세요.


!pip install openpyxl rdkit-pypi torch torch_geometric pandas


In [1]:
import pandas as pd
from rdkit import Chem
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

In [2]:
df = pd.read_excel('LOHC_data.xlsx', engine='openpyxl')
print("Columns:", df.columns.tolist())

# 기준 SMILES 및 타겟 열 이름
de_smiles_col = 'Dehydrogenated_SMILES'
hy_smiles_col = 'Hydrogenated_SMILES'
label_cols = ['Dehydrogenated_energy', 'Hydrogenated_energy', 'Potential']

Columns: ['Dehydrogenated_SMILES', 'Hydrogenated_SMILES', 'Dehydrogenated_energy', 'Hydrogenated_energy', 'Potential', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17']


In [9]:
def smiles_to_graph(smiles: str, labels: list) -> Data:
    mol = Chem.MolFromSmiles(smiles)
    nodes = []
    for atom in mol.GetAtoms():
        nodes.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetTotalNumHs(),
            atom.GetFormalCharge(),
            int(atom.GetIsAromatic())
        ])
    x = torch.tensor(nodes, dtype=torch.float)

    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        a, b = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        for i, j in ((a,b),(b,a)):
            edge_index.append([i, j])
            edge_attr.append([bond.GetBondTypeAsDouble()])

    edge_index = torch.tensor(edge_index).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    y = torch.tensor([labels], dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

In [10]:
data_list = []
for _, row in df.iterrows():
    labels = row[label_cols].tolist()
    for col in (de_smiles_col, hy_smiles_col):
        g = smiles_to_graph(row[col], labels)
        data_list.append(g)

print(f"Total graphs: {len(data_list)}")

Total graphs: 616


In [11]:
batch_size = 16
loader = DataLoader(data_list, batch_size=batch_size, shuffle=True)

C:\Users\user\AppData\Roaming\Python\Python310\site-packages\torch_geometric\deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [12]:
class GCN(torch.nn.Module):
    def __init__(self, in_feats, hidden_feats, out_feats):
        super().__init__()
        self.conv1 = GCNConv(in_feats, hidden_feats)
        self.conv2 = GCNConv(hidden_feats, hidden_feats)
        self.lin   = torch.nn.Linear(hidden_feats, out_feats)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.lin(x)

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(
    in_feats = data_list[0].num_node_features,
    hidden_feats = 64,
    out_feats = len(label_cols)
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

In [14]:
epochs = 50
for epoch in range(1, epochs+1):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        loss = criterion(pred, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    avg_loss = total_loss / len(loader.dataset)
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}, Loss: {avg_loss:.4f}")

Epoch 001, Loss: nan
Epoch 010, Loss: nan
Epoch 020, Loss: nan
Epoch 030, Loss: nan
Epoch 040, Loss: nan
Epoch 050, Loss: nan


In [15]:
model.eval()
preds = []
with torch.no_grad():
    for batch in loader:
        batch = batch.to(device)
        out = model(batch)
        preds.append(out.cpu())
preds = torch.cat(preds, dim=0).numpy()

# 예측 결과를 DataFrame으로 정리
pred_df = pd.DataFrame(preds, columns=label_cols)
print(pred_df.head())

   Dehydrogenated_energy  Hydrogenated_energy  Potential
0                    NaN                  NaN        NaN
1                    NaN                  NaN        NaN
2                    NaN                  NaN        NaN
3                    NaN                  NaN        NaN
4                    NaN                  NaN        NaN


In [ ]:
pred_df.to_csv('predictions.csv', index=False)
print("Saved predictions.csv")